# Correct synthetic photometry errors

GaiaXPy provides the function `apply_error_correction` to apply tabulated corrections to flux errors for supported photometric systems.

The correction method is described in the DR3 synthetic photometry paper by [Gaia Collaboration, Montegriffo et al. (2023)](https://doi.org/10.1051/0004-6361/202243709). GaiaXPy provides DR3 correction factors for a subset of the supported systems.

The function `generate` can also apply the corrections while computing photometry from XP spectra. This is recommended. See [Corrections during generation](#gaiaxpy-corrections-during-generation).

**Contents**

- [Input photometry](#gaiaxpy-input-photometry)
- [Flux-error correction](#gaiaxpy-flux-error-correction)
- [Corrections during generation](#gaiaxpy-corrections-during-generation)
- [Systems without correction tables](#gaiaxpy-systems-without-correction-tables)
- [System selection and output files](#gaiaxpy-system-selection-and-output-files)
- [Correction metadata and reference](#gaiaxpy-correction-metadata)

<a id="gaiaxpy-input-photometry" name="gaiaxpy-input-photometry"></a>

## Input photometry

The input expected by `apply_error_correction` is an uncorrected photometry DataFrame with the Gaia Vega $G$ magnitude included among its columns. The table must include `source_id` and the flux-error columns to be corrected. This example uses the function `generate` with `error_correction=False` and produces DR3 JKC and Gaia Vega photometry together.

**In order to run this tutorial, replace `spectra_input` with an existing file.**

In [ ]:
from gaiaxpy import PhotometricSystem, apply_error_correction, generate

spectra_input = "path/to/dr3_xp_mean_spectra.ecsv"
systems = [PhotometricSystem.DR3_JKC, PhotometricSystem.DR3_Gaia_Vega]
photometry = generate(spectra_input, photometric_system=systems, error_correction=False, save_file=False)

<a id="gaiaxpy-flux-error-correction" name="gaiaxpy-flux-error-correction"></a>

## Flux-error correction

With no system selection, GaiaXPy infers the systems in the input data from its columns; alternatively `photometric_system` can be used to provide the systems explicitly. The example compares the original and corrected flux errors. GaiaXPy multiplies the supported flux errors by factors selected using the Gaia Vega $G$ magnitude; therefore, the Gaia Vega system must be included in the input. Each call to `apply_error_correction` applies the factors again, so the input table provided to this function must be uncorrected.

In [ ]:
corrected = apply_error_correction(photometry, save_file=False)

# Alternatively with explicit photometric_system:
# corrected = apply_error_correction(photometry, photometric_system=systems, save_file=False)

import pandas as pd

column = "DR3_Jkc_flux_error_V"
pd.DataFrame({"source_id": photometry["source_id"],
              "original_flux_error_V": photometry[column],
              "corrected_flux_error_V": corrected[column]}).head()

<a id="gaiaxpy-corrections-during-generation" name="gaiaxpy-corrections-during-generation"></a>

## Corrections during generation

For continuous XP spectra, `generate` can apply the flux-error correction directly.

In this example, only DR3 JKC is requested. The function `generate` computes the Gaia Vega reference internally and then discards it.

In [ ]:
generated = generate(spectra_input, photometric_system=PhotometricSystem.DR3_JKC, error_correction=True, save_file=False)

<a id="gaiaxpy-systems-without-correction-tables" name="gaiaxpy-systems-without-correction-tables"></a>

## Systems without correction tables

Correction factors are available only for selected photometric systems and passbands in each release. GaiaXPy issues a warning and leaves the corresponding flux errors unchanged when a system has no correction table or a passband has no factors in its system's table. Other requested systems and passbands with factors are still corrected.

DR3 LSST has no correction table. The following example requests the correction for DR3 JKC and LSST together. GaiaXPy corrects the JKC flux errors and reports:

```text
UserWarning: System DR3_Lsst does not have a correction table. The program will not apply error correction over this system.
```

In [ ]:
mixed_photometry = generate(
    spectra_input, photometric_system=[PhotometricSystem.DR3_JKC, PhotometricSystem.DR3_LSST],
    error_correction=True, save_file=False
)

<a id="gaiaxpy-system-selection-and-output-files" name="gaiaxpy-system-selection-and-output-files"></a>

## System selection and output files

An explicit selection must include the Gaia Vega system for the release alongside the systems selected for correction. Gaia Vega acts as a reference for the error correction. The example takes an uncorrected input table and saves the corrected result in ECSV.

In [ ]:
corrected = apply_error_correction(
    photometry, photometric_system=systems,
    output_file="corrected_photometry", output_format="ecsv", save_file=True
)

<a id="gaiaxpy-correction-metadata" name="gaiaxpy-correction-metadata"></a>

## Correction metadata and reference

For traceability purposes, GaiaXPy stores provenance in the DataFrame attribute `gaiaxpy_metadata`. A standalone call to `apply_error_correction` appends an entry to `operations`, preserving the provenance of the original photometry. The correction record for that operation identifies the tables and passbands used. A status of `applied` means that at least one passband was corrected. The `systems` field lists the systems and passbands actually corrected. When correction is requested directly through `generate`, the record is stored under the top-level `error_correction` key.

The example below uses the `corrected` DataFrame produced by `apply_error_correction`.

In [ ]:
metadata = corrected.attrs["gaiaxpy_metadata"]
correction = metadata["operations"][0]["error_correction"]

print("Status:", correction["status"])
print()
for entry in correction["files"]:
    details = entry["metadata"]
    print("Correction table:", entry["file"])
    print("Gaia release:", details.get("data_release"))
    print("Reference:", details.get("reference", "No reference recorded"))
    print()

The correction tables cite the DR3 synthetic photometry paper for the method used to derive the factors. The table metadata gives the release to which the factors apply. [Gaia Collaboration, Montegriffo et al. (2023)](https://doi.org/10.1051/0004-6361/202243709).

For a result produced directly by `generate` with `error_correction=True`, the same table and reference fields are available under `generated.attrs["gaiaxpy_metadata"]["error_correction"]`, where `generated` is the returned DataFrame. The metadata can also be read from the ECSV file in the preceding example.

In [ ]:
from gaiaxpy.output import read_output_metadata

# This output file is available after running the ECSV save example.
saved_metadata = read_output_metadata("corrected_photometry.ecsv")
saved_metadata["operations"][0]["error_correction"]["files"]